# 导包

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import toollib as tl
import polars as pl
import pandas as pd
from datetime import datetime,timedelta
import time
import glob
import os

In [ ]:
sys_name = 'mw'
user,passwd = 'etl','Vrrbpf6PjYStx4+v'
work_dir = '/home/model/mw'
sample_dir = os.path.join(work_dir,'sample')
req_dir = os.path.join(work_dir,'req_dir')

models = [
    'app_mx_loan_v1',"app_mx_base_v1","app_mx_comp_v1","app_mx_gcate_v1",
    'sms_mx_comp_v1','sms_mx_base_v1','sms_mx_rule_v1',
    'loan_mx_base_v2','loan_mx_base_v3_2',
    'call_mx_base0_v1',
    'cross_mx_base_v1',
    'dev_mx_base_v1','dev_mx_base_v2',
    'ui_mx_base_v1','ui_mx_base_v2',
]

max_workers = 30
port = 1102

featurelib_path = '~/featurelib/'
python_bin='~/miniconda3/envs/py310/bin/'

write_mode = 'increament' #'over_witre' # increament 覆盖写还是增量写
sample_start_time = datetime.now() - timedelta(days=210)
sample_start_time = sample_start_time.strftime('%Y-%m-%d')
sample_start_time = '2024-07-01'
sample_end_time = datetime.now() - timedelta(days=1)
sample_end_time = sample_end_time.strftime('%Y-%m-%d')
print('word_dir:',work_dir)
print('sample_dir',sample_dir)
print('req_dir',req_dir)
print(sample_start_time,sample_end_time)

In [ ]:
partition_cols=['new_old_user_status','loan_date']

# 查询并生成样本信息

In [ ]:
loan_df = tl.get_loans(sys_name,user,passwd,start_date=sample_start_time,end_date=sample_end_time)
loan_df['loan_date'] = loan_df['create_time'].map(lambda x: str(x)[0:10])

In [ ]:
sample_df = loan_df[(loan_df['extension_count']==0)&(loan_df.device_type=='android')&(loan_df['installment_num']==1)]
sample_df = sample_df.sort_values(by='loan_time')
sample_df = sample_df.drop_duplicates(subset='app_order_id')
print(f"文件大小:{sample_df.shape},订单枚举:{sample_df['app_order_id'].nunique()},max_loan_time:{sample_df['loan_time'].max()}")
sample_df.to_parquet(sample_dir,partition_cols=partition_cols,compression='zstd',existing_data_behavior='delete_matching')
print(sample_df['new_old_user_status'].value_counts())

# 下载基本的样本信息

In [ ]:
sample_df = pd.read_parquet(sample_dir)
print(f"文件大小:{sample_df.shape},订单枚举:{sample_df['app_order_id'].nunique()},max_loan_time:{sample_df['loan_time'].max()}")

In [ ]:
sample_paths = glob.glob(f'{sample_dir}/**/*.parquet',recursive=True)
sample_partitions_all =  dict(  tl.partition_path_parser(p) for p in sample_paths )
sample_partitions = {}
for k,v in sample_partitions_all.items():
    loan_date = v['loan_date']
    if (loan_date>= sample_start_time) and (loan_date< sample_end_time):
        sample_partitions[k] = v
        
print(f"全部的分区数为:{len(sample_partitions_all)},本次任务的分区数为:{len(sample_partitions)}")

In [ ]:
def fetch_datas(sample_dir,req_dir,new_old_user_status,loan_date,partition_path):
    print(f"开始处理:{partition_path}")
    df = pd.read_parquet(sample_dir,filters=[
            ('new_old_user_status','=', new_old_user_status ),
            ('loan_date','=',loan_date)
        ])
    orderlist = df['app_order_id'].to_list()
    req_df = tl.get_sample_req(orderlist,sys_name,user,passwd)
    req_df['new_old_user_status'] = new_old_user_status
    req_df['loan_date'] = loan_date
    req_df['ctime']  = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    req_df.to_parquet(req_dir,partition_cols=partition_cols,compression='zstd',existing_data_behavior='delete_matching')
    
for k,v in sample_partitions.items():
    partition_path = f"{req_dir}/{k}"
    new_old_user_status  = int(v['new_old_user_status'])
    loan_date = v['loan_date']
    
    if write_mode == 'increament':
        try:
            tmp = pl.read_parquet(partition_path)
            print(f'{partition_path}分区数据已经存在，无需下载! {tmp.shape}')
        except:
            fetch_datas(sample_dir,req_dir,new_old_user_status,loan_date,partition_path)
    elif write_mode == 'over_witre':
        fetch_datas(sample_dir,req_dir,new_old_user_status,loan_date,partition_path)
    else:
        raise ValueError(f'write_mode 参数必须是 over_witre 或 increament')

# 回溯特征

In [ ]:
tl.stop_ser(port)
print(f"服务和特征模块的运行关系：")
feature_maps = tl.parse_mode_name(model_names=models)
for k,v in feature_maps.items():
    print(k,v)
time.sleep(2)

In [ ]:
req_paths = glob.glob(f'{req_dir}/**/*.parquet',recursive=True)
req_partitions_all =  dict(  tl.partition_path_parser(p) for p in req_paths )
req_partitions = {}
for k,v in req_partitions_all.items():
    loan_date = v['loan_date']
    if (loan_date>= sample_start_time) and (loan_date< sample_end_time):
        req_partitions[k] = v
print(f"全部的分区数为:{len(req_partitions_all)},本次任务的分区数为:{len(req_partitions)}")

In [ ]:
def calc_features(req_dir,model,partition_path,new_old_user_status,loan_date,uri,max_workers):
    req_df = pd.read_parquet(req_dir,filters=[
      ('new_old_user_status','=',new_old_user_status),
      ('loan_date','=',loan_date)
    ])
    result = tl.request_featurelib(req_df,model,uri=uri,max_workers=max_workers,tqdm_desc=partition_path)
    result['new_old_user_status'] = new_old_user_status
    result['loan_date'] = loan_date
    result['ctime'] = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    save_path = os.path.join(work_dir,model)
    result.to_parquet(save_path,partition_cols=partition_cols,compression='zstd',existing_data_behavior='delete_matching')

for ser_name,model_list in feature_maps.items():
    print(f"启动{ser_name}的服务")
    tl.start_ser(ser_name,port=port,workers=max_workers,featurelib_path=featurelib_path,python_bin=python_bin)
    uri = f'http://127.0.0.1:{port}'
    for model in model_list:
        for k,v in req_partitions.items():
            partition_path = os.path.join(work_dir,f'{model}/{k}') #f'{model}/{k}'
            new_old_user_status  = int(v['new_old_user_status'])
            loan_date = v['loan_date']
            if write_mode == 'increament':
                try:
                    tmp = pl.read_parquet(partition_path)
                    print(f'{partition_path} 已经存在,无需计算! {tmp.shape}')
                except:
                    calc_features(req_dir,model,partition_path,new_old_user_status,loan_date,uri,max_workers)
            elif write_mode == 'over_witre':
                calc_features(req_dir,model,partition_path,new_old_user_status,loan_date,uri,max_workers)
            else:
                raise ValueError(f'write_mode 参数必须是 over_witre 或 increament')
    print(f'开始关闭{ser_name}')
    tl.stop_ser(port)
    time.sleep(2)

## 单独跑不能开大量进程的sms_mx_loan特征

In [ ]:
models = [
    'sms_mx_loan_v1'
]
max_workers = 4

In [ ]:
tl.stop_ser(port)
print(f"服务和特征模块的运行关系：")
feature_maps = tl.parse_mode_name(model_names=models)
for k,v in feature_maps.items():
    print(k,v)
time.sleep(2)
def calc_features(req_dir,model,partition_path,new_old_user_status,loan_date,uri,max_workers):
    req_df = pd.read_parquet(req_dir,filters=[
      ('new_old_user_status','=',new_old_user_status),
      ('loan_date','=',loan_date)
    ])
    result = tl.request_featurelib(req_df,model,uri=uri,max_workers=max_workers,tqdm_desc=partition_path)
    result['new_old_user_status'] = new_old_user_status
    result['loan_date'] = loan_date
    result['ctime'] = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    save_path = os.path.join(work_dir,model)
    result.to_parquet(save_path,partition_cols=partition_cols,compression='zstd',existing_data_behavior='delete_matching')

for ser_name,model_list in feature_maps.items():
    print(f"启动{ser_name}的服务")
    tl.start_ser(ser_name,port=port,workers=max_workers,featurelib_path=featurelib_path,python_bin=python_bin)
    uri = f'http://127.0.0.1:{port}'
    for model in model_list:
        for k,v in req_partitions.items():
            partition_path = os.path.join(work_dir,f'{model}/{k}') #f'{model}/{k}'
            new_old_user_status  = int(v['new_old_user_status'])
            loan_date = v['loan_date']
            if write_mode == 'increament':
                try:
                    tmp = pl.read_parquet(partition_path)
                    print(f'{partition_path} 已经存在,无需计算! {tmp.shape}')
                except:
                    calc_features(req_dir,model,partition_path,new_old_user_status,loan_date,uri,max_workers)
            elif write_mode == 'over_witre':
                calc_features(req_dir,model,partition_path,new_old_user_status,loan_date,uri,max_workers)
            else:
                raise ValueError(f'write_mode 参数必须是 over_witre 或 increament')
    print(f'开始关闭{ser_name}')
    tl.stop_ser(port)
    time.sleep(2)

In [ ]:
"""
jupyter nbconvert --to script req_and_fea_mw.ipynb
conda activate py310
python req_and_fea_am.py
chmod -R +777 /home/model/
"""